In [ ]:
!pip install medmnist
!pip install torch torchvision scikit-learn tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.3 MB/s eta 0:00:00


In [2]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
import time

import medmnist
from medmnist import INFO, Evaluator
from torchvision import datasets, transforms, models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#data_flag = 'pathmnist'

## You can tune these three hyperparameters
NUM_EPOCHS = 10
BATCH_SIZE = 128
LR = 3e-4

In [3]:
class MedMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None):
        # Load the compressed numpy file
        data = np.load(npz_path)

        # Access images and labels based on the specific split
        self.images = data[f'{split}_images']
        self.labels = data[f'{split}_labels'].flatten()
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx] # Shape (28, 28)

        # 1. Prepare for Resize (must be H, W, C for Resize/ToTensor)
        if len(image.shape) == 2:
            image = np.stack([image] * 3, axis=-1)

        # 2. Apply Transform
        # Resize -> ToTensor (scales to 0-1) -> Normalize
        if self.transform:
            image = self.transform(image)
        else:
            # Fallback if no transform: manual CHW swap and float conversion
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        # 3. Label must be Long for CrossEntropyLoss
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return image, label


In [8]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
npz_file_path = "/content/gdrive/MyDrive/csen240-w26/pathmnist_subset.npz"

base_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

aug_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((28, 28)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

Mounted at /content/gdrive


In [9]:
print(f"Device: {device}")

train_ds = MedMNISTDataset(npz_file_path, split='train', transform=aug_transform)
val_ds = MedMNISTDataset(npz_file_path, split='val', transform=base_transform)
train_loader_1 = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader_1 = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(np.unique(train_ds.labels))
print(f"Num classes: {num_classes}")

criterion = nn.CrossEntropyLoss() # Combines LogSoftmax and NLLLoss

Device: cuda
Num classes: 9


In [10]:
import time
import torch
from tqdm import tqdm
from sklearn.metrics import f1_score

def get_metrics(loader, model, device, criterion):
    model.eval()
    all_preds, all_labels = [], []
    all_loss = 0.0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device).long().view(-1)

            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            all_loss += loss.item()

            preds = outputs.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(lbls.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    avg_loss = all_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average="macro")
    acc = 100.0 * (all_preds == all_labels).mean()
    return f1, acc, avg_loss


def get_param_groups(model, lr):
    if hasattr(model, "features") and hasattr(model, "head"):
        return [
            {"params": model.features.parameters(), "lr": lr / 3},
            {"params": model.head.parameters(), "lr": lr},
        ]

    # Case 2: ResNet-style model with .fc
    elif hasattr(model, "fc"):
        backbone_params = []
        head_params = list(model.fc.parameters())
        head_param_ids = {id(p) for p in head_params}

        for p in model.parameters():
            if id(p) not in head_param_ids:
                backbone_params.append(p)

        return [
            {"params": backbone_params, "lr": lr / 3},
            {"params": head_params, "lr": lr},
        ]

    else:
        return [{"params": model.parameters(), "lr": lr}]


def run_training(train_loader, val_loader, model, device, criterion, LR, NUM_EPOCHS):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        get_param_groups(model, LR),
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS
    )

    print(f"Training initiated on {device}...")

    for epoch in range(NUM_EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0

        for images, labels in tqdm(train_loader):
            images = images.to(device)
            labels = labels.to(device).long().view(-1)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        scheduler.step()

        epoch_duration = time.time() - start_time
        avg_train_loss = running_loss / len(train_loader)

        train_f1, train_acc, _ = get_metrics(train_loader, model, device, criterion)
        val_f1, val_acc, val_loss = get_metrics(val_loader, model, device, criterion)

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Duration: {epoch_duration:.2f}s")
        print(f"  [TRAIN] Loss: {avg_train_loss:.4f} | F1: {train_f1:.4f} | Acc: {train_acc:.2f}%")
        print(f"  [VAL]   Loss: {val_loss:.4f} | F1: {val_f1:.4f} | Acc: {val_acc:.2f}%")

In [11]:
import torch
import torch.nn as nn
from torchvision import models

class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim, out_dim, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(p)
        self.fc2 = nn.Linear(out_dim, out_dim)

        self.skip = nn.Identity() if in_dim == out_dim else nn.Linear(in_dim, out_dim)

        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        residual = self.skip(x)
        out = self.fc1(x)
        out = self.act(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = out + residual
        out = self.norm(out)
        out = self.act(out)
        return out

In [12]:
class CustomNN(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()

        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.efficientnet_v2_s(weights=weights)

        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.features = backbone

        self.norm = nn.LayerNorm(in_features)
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(0.2),

            ResidualMLPBlock(256, 256, p=0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = self.head(x)
        return x

model = CustomNN(num_classes)
run_training(train_loader_1, val_loader_1, model, device, criterion, LR, NUM_EPOCHS=16)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 190MB/s]


Training initiated on cuda...


100%|██████████| 704/704 [02:36<00:00,  4.49it/s]


Epoch [1/16] Duration: 156.80s
  [TRAIN] Loss: 1.0316 | F1: 0.7695 | Acc: 77.16%
  [VAL]   Loss: 0.5349 | F1: 0.8069 | Acc: 80.88%


...

Epoch [14/16] Duration: 154.01s
  [TRAIN] Loss: 0.1713 | F1: 0.9503 | Acc: 94.97%
  [VAL]   Loss: 0.1127 | F1: 0.9616 | Acc: 96.12%


100%|██████████| 704/704 [02:32<00:00,  4.63it/s]


Epoch [15/16] Duration: 152.14s
  [TRAIN] Loss: 0.1692 | F1: 0.9517 | Acc: 95.10%
  [VAL]   Loss: 0.1114 | F1: 0.9620 | Acc: 96.16%


100%|██████████| 704/704 [02:32<00:00,  4.61it/s]


Epoch [16/16] Duration: 152.71s
  [TRAIN] Loss: 0.1651 | F1: 0.9515 | Acc: 95.09%
  [VAL]   Loss: 0.1102 | F1: 0.9629 | Acc: 96.26%


In [13]:
# Using ResNet-18 as the baseline architecture
model_2 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_2.fc = nn.Linear(model_2.fc.in_features, num_classes) # Adjust output layer for medical classes

run_training(train_loader_1, val_loader_1, model_2, device, criterion, LR, NUM_EPOCHS=16)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


Training initiated on cuda...


100%|██████████| 704/704 [01:54<00:00,  6.13it/s]


Epoch [1/16] Duration: 114.83s
  [TRAIN] Loss: 0.7265 | F1: 0.8183 | Acc: 81.84%
  [VAL]   Loss: 0.3886 | F1: 0.8610 | Acc: 86.01%


...

Epoch [14/16] Duration: 113.00s
  [TRAIN] Loss: 0.2204 | F1: 0.9282 | Acc: 92.73%
  [VAL]   Loss: 0.1678 | F1: 0.9412 | Acc: 94.05%


100%|██████████| 704/704 [01:55<00:00,  6.11it/s]


Epoch [15/16] Duration: 115.22s
  [TRAIN] Loss: 0.2179 | F1: 0.9294 | Acc: 92.85%
  [VAL]   Loss: 0.1617 | F1: 0.9443 | Acc: 94.35%


100%|██████████| 704/704 [01:53<00:00,  6.21it/s]


Epoch [16/16] Duration: 113.29s
  [TRAIN] Loss: 0.2132 | F1: 0.9294 | Acc: 92.86%
  [VAL]   Loss: 0.1620 | F1: 0.9417 | Acc: 94.09%


In [14]:
PATH = '/content/gdrive/MyDrive/csen240-w26/my_model1.pt'
torch.save(model.state_dict(), PATH)


In [16]:
PATH = '/content/gdrive/MyDrive/csen240-w26/my_model2.pt'
torch.save(model_2.state_dict(), PATH)


In [20]:
PATH_TEST = '/content/gdrive/MyDrive/csen240-w26/pathmnist_test.npz'
test_ds = MedMNISTDataset(npz_file_path, split='val', transform=base_transform)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [22]:
test_f1, test_acc, test_loss = get_metrics(test_loader, model, device, criterion)
print("Custom Architecture F1: ", test_f1)

Custom Architecture F1:  0.9628875882685074


In [24]:
test_f1_base, test_acc_base, test_loss_base = get_metrics(test_loader, model_2, device, criterion)
print("Base Architecture F1: ", test_f1_base)

Base Architecture F1:  0.9416758898985536
